# HMM Training mit CmdStanPy (Parallel)

Dieses Notebook trainiert **nur HMM** Modelle mit S=2,3,4.

**Vorteile von CmdStanPy:**

- ✓ Chains laufen parallel (statt sequenziell)
- ✓ ~2x schneller bei 2 Chains
- ✓ Bessere Performance

**Wichtig:** Lassen Sie parallel das VD-HMM-Notebook laufen für maximale Effizienz!


In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path("..").resolve()))

import pickle
import numpy as np
from helpers import ModelData
import cmdstanpy

print(f"CmdStanPy Version: {cmdstanpy.__version__}")
print(f"CmdStan Path: {cmdstanpy.cmdstan_path()}")

CmdStanPy Version: 1.3.0
CmdStan Path: /Users/omidsedighi-mornani/.cmdstan/cmdstan-2.37.0


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load processed data
data_path = "../data/processed/processed_data.pkl"
model_data = ModelData.from_pickle(data_path)

print(model_data.summary())


ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 200
- Eval indices: 221

Benchmark data: Available
- Benchmark covariates shape: (921, 23)
- Columns: business_id, density, Checkin, category, chain...



In [3]:
# Configuration
stan_model_folder = Path("../data/stan_code")
fitted_model_folder = Path("../models")
fitted_model_folder.mkdir(parents=True, exist_ok=True)

print(f"✓ Configuration set")
print(f"  Stan models: {stan_model_folder}")
print(f"  Output folder: {fitted_model_folder}")

✓ Configuration set
  Stan models: ../data/stan_code
  Output folder: ../models


In [4]:
def prepare_stan_data(model_data: ModelData, S: int):
    """
    Bereitet die Daten für Stan vor.

    CmdStanPy braucht die Daten als JSON-kompatibles Dictionary.
    """
    stan_data = {
        "S": S,
        "N_total": int(model_data.n_total),
        "N_train": int(model_data.n_train),
        "N_obs": int(model_data.n_obs),
        "nCovs": int(model_data.n_covs),
        "Time": [int(x) for x in model_data.time],
        "Closed": [int(x) for x in model_data.closed],
        "Days": [float(x) for x in model_data.days],
        "Ratings": [int(x) for x in model_data.ratings],
        "Sentiment": [float(x) for x in model_data.sentiment],
        "Q": model_data.Q.tolist(),
        "R": model_data.R.tolist(),
        "X_test": model_data.X_test.tolist(),
    }
    return stan_data


# Test
test_data = prepare_stan_data(model_data, S=3)
print(f"✓ Stan data prepared")
print(f"  Keys: {list(test_data.keys())}")

✓ Stan data prepared
  Keys: ['S', 'N_total', 'N_train', 'N_obs', 'nCovs', 'Time', 'Closed', 'Days', 'Ratings', 'Sentiment', 'Q', 'R', 'X_test']


In [ ]:
def train_model_cmdstan(
    model_data,
    S,
    model_name="hmm",
    chains=2,
    parallel_chains=2,
    iter_warmup=500,
    iter_sampling=500,
    seed=42,
    adapt_delta=0.8,
    max_treedepth=10,
):
    """
    Trainiert ein Modell mit CmdStanPy.

    Parameters:
    -----------
    model_data : ModelData
        Daten für das Training
    S : int
        Anzahl der Hidden States (2-5)
    model_name : str
        'vdhmm' oder 'hmm'
    chains : int
        Anzahl der MCMC Chains
    parallel_chains : int
        Anzahl parallel laufender Chains (nutzt parallel_chains CPU Cores)
    iter_warmup : int
        Warmup Iterationen
    iter_sampling : int
        Sampling Iterationen (post-warmup)
    seed : int
        Random Seed
    adapt_delta : float
        Stan adapt_delta Parameter (0.8-0.99, höher = konservativer)
    max_treedepth : int
        Stan max_treedepth Parameter

    Returns:
    --------
    cmdstanpy.CmdStanMCMC : Fit-Objekt
    """
    assert model_name in ["vdhmm", "hmm"], f"Invalid model_name: {model_name}"
    assert S in range(2, 6), "S must be between 2 and 5"

    # Daten vorbereiten
    stan_data = prepare_stan_data(model_data, S)

    # Model file
    model_file = stan_model_folder / f"{model_name}.stan"
    if not model_file.exists():
        raise FileNotFoundError(f"Stan model not found: {model_file}")

    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} with S={S} states (CmdStanPy)")
    print(f"{'='*60}")
    print(f"Model file: {model_file}")
    print(f"\nConfiguration:")
    print(f"  Chains: {chains}")
    print(f"  Parallel chains: {parallel_chains}")
    print(f"  Warmup iterations: {iter_warmup}")
    print(f"  Sampling iterations: {iter_sampling}")
    print(f"  Total iterations: {iter_warmup + iter_sampling}")
    print(f"  Seed: {seed}")
    print(f"  Adapt delta: {adapt_delta}")
    print(f"  Max treedepth: {max_treedepth}")

    # Compile model
    print(f"\nCompiling model...")
    model = cmdstanpy.CmdStanModel(stan_file=str(model_file))
    print(f"✓ Model compiled")

    # Sample
    print(f"\nSampling...")
    fit = model.sample(
        data=stan_data,
        chains=chains,
        parallel_chains=parallel_chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
        show_progress=True,
    )

    print(f"\n✓ Sampling complete!")

    # Save model
    output_path = fitted_model_folder / f"{model_name}_{S}_cmdstan.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(
            {
                "fit": fit,
                "model_name": model_name,
                "S": S,
                "stan_data": stan_data,
                "summary": fit.summary(),
            },
            f,
        )

    print(f"✓ Model saved to {output_path}")

    # Diagnostics
    print(f"\n{'-'*60}")
    print("Diagnostics:")
    print(f"{'-'*60}")
    print(fit.diagnose())

    # Summary statistics
    print(f"\n{'-'*60}")
    print("Summary (first 20 parameters):")
    print(f"{'-'*60}")
    summary_df = fit.summary()
    print(summary_df.head(20))

    return fit


print("✓ Training function defined (CmdStanPy)")

✓ Training function defined (CmdStanPy)


## Training Configuration

**Paper-Standard:** 2 chains × 1000 iterations (500 warmup + 500 sampling)

**Für schnelles Testen:** 2 chains × 200 iterations (100 warmup + 100 sampling)


In [6]:
# Training Settings
SEED = 42
CHAINS = 4
PARALLEL_CHAINS = 4  # Nutzt 4 CPU Cores parallel
ITER_WARMUP = 500  # Paper: 500, Quick test: 100
ITER_SAMPLING = 500  # Paper: 500, Quick test: 100
ADAPT_DELTA = 0.8  # 0.8-0.95, höher falls divergent transitions
MAX_TREEDEPTH = 10  # 10-15

np.random.seed(SEED)

print("Training Configuration:")
print(f"  Seed: {SEED}")
print(f"  Chains: {CHAINS}")
print(f"  Parallel chains: {PARALLEL_CHAINS}")
print(f"  Warmup iterations: {ITER_WARMUP}")
print(f"  Sampling iterations: {ITER_SAMPLING}")
print(f"  Total iterations: {ITER_WARMUP + ITER_SAMPLING}")
print(f"  Total posterior samples: {CHAINS * ITER_SAMPLING}")

Training Configuration:
  Seed: 42
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 500
  Sampling iterations: 500
  Total iterations: 1000
  Total posterior samples: 2000


## Training HMM Models (S=2, 3, 4)

Standard Hidden Markov Models mit zeitunabhängigen Übergangswahrscheinlichkeiten.


In [ ]:
# Dictionary zum Speichern aller Modelle
trained_models = {}

# HMM Training für S=2 bis S=4
for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model_cmdstan(
            model_data=model_data,
            S=S,
            model_name="hmm",
            chains=CHAINS,
            parallel_chains=PARALLEL_CHAINS,
            iter_warmup=ITER_WARMUP,
            iter_sampling=ITER_SAMPLING,
            seed=SEED,
            adapt_delta=ADAPT_DELTA,
            max_treedepth=MAX_TREEDEPTH,
        )

        trained_models[f"hmm_{S}"] = fit
        print(f"\n✓✓✓ HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("HMM Training Complete!")
print("=" * 60)

00:22:44 - cmdstanpy - INFO - compiling stan file /var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/tmpueggga7x/tmp86t3wcry.stan to exe file /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/stan_code/hmm




############################################################
# HMM Training: S=2
############################################################


Training HMM with S=2 states (CmdStanPy)
Model file: ../data/stan_code/hmm.stan

Configuration:
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 500
  Sampling iterations: 500
  Total iterations: 1000
  Seed: 42
  Adapt delta: 0.8
  Max treedepth: 10

Compiling model...


00:23:00 - cmdstanpy - INFO - compiled model executable: /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/stan_code/hmm


✓ Model compiled

Sampling...


00:23:01 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]


chain 1:   0%|          | 1/1000 [00:00<06:40,  2.49it/s, (Warmup)]







chain 1:  10%|█         | 100/1000 [10:21<1:33:32,  6.24s/it, (Warmup)]



chain 1:  20%|██        | 200/1000 [15:18<57:30,  4.31s/it, (Warmup)]  


chain 1:  30%|███       | 300/1000 [18:25<37:16,  3.19s/it, (Warmup)]








chain 1:  40%|████      | 400/1000 [21:31<26:40,  2.67s/it, (Warmup)]




chain 1:  50%|█████     | 501/1000 [24:37<19:40,  2.37s/it, (Sampling)]


chain 1:  60%|██████    | 600/1000 [27:25<13:54,  2.09s/it, (Sampling)]




chain 2: 100%|██████████| 1000/1000 [37:26<00:00,  2.25s/it, (Sampling completed)]

chain 3: 100%|██████████| 1000/1000 [37:26<00:00,  2.25s/it, (Sampling completed)]


chain 4: 100%|██████████| 1000/1000 [37:26<00:00,  2.25s/it, (Sampling completed)]

...


✓ Sampling complete!
✓ Model saved to ../models/hmm_3_cmdstan.pkl

------------------------------------------------------------
Diagnostics:
------------------------------------------------------------
Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
6 of 2000 (0.30%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
The E-BFMI, 0.01, is below the nominal threshold of 0.30 which suggests that HMC may have trouble exploring the target distribution.
If possible, try to reparameterize the model.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:


02:47:31 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]



chain 1:   0%|          | 1/1000 [00:02<35:33,  2.14s/it, (Warmup)]

chain 1:  30%|███       | 300/1000 [1:40:10<3:14:46, 16.70s/it, (Warmup)]

chain 1:  40%|████      | 400/1000 [1:50:39<2:05:50, 12.58s/it, (Warmup)]


chain 1:  50%|█████     | 501/1000 [2:02:25<1:27:42, 10.55s/it, (Sampling)]





chain 1:  60%|██████    | 600/1000 [2:14:02<1:00:27,  9.07s/it, (Sampling)]


chain 1:  70%|███████   | 700/1000 [2:25:31<41:14,  8.25s/it, (Sampling)]  







chain 1:  80%|████████  | 800/1000 [2:37:14<26:03,  7.82s/it, (Sampling)]


chain 1: 100%|██████████| 1000/1000 [2:47:14<00:00,  5.12s/it, (Sampling)]








chain 2: 100%|██████████| 1000/1000 [3:02:47<00:00, 10.97s/it, (Sampling completed)]

chain 3: 100%|██████████| 1000/1000 [3:02:47<00:00, 10.97s/it, (Sampling completed)]


chain 4: 100%|██████████| 1000/1000 [3:02:47<00:00, 10.97s/it, (Sampling completed)]


05:50:19 - cmdstanpy - INFO - CmdStan done processing.
05:50:19 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is inf, but should be greater than the previous element, inf (in 'hmm.stan', line 172, column 8 to column 74)
	Exception: ordered_probit: Cut-points is not a valid ordered vector. The element at 2 is i

05:50:20 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 63 divergent transitions (12.6%)
	Chain 2 had 13 divergent transitions (2.6%)
	Chain 3 had 16 divergent transitions (3.2%)
	Chain 4 had 20 divergent transitions (4.0%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.



✓ Sampling complete!
✓ Model saved to ../models/hmm_4_cmdstan.pkl

------------------------------------------------------------
Diagnostics:
------------------------------------------------------------
Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
112 of 2000 (5.60%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
The E-BFMI, 0.01, is below the nominal threshold of 0.30 which suggests that HMC may have trouble exploring the target distribution.
If possible, try to reparameterize the model.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01

## Training Summary


In [ ]:
# Summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"\nSaved in: {fitted_model_folder}")

# List saved models
saved_models = sorted(fitted_model_folder.glob("hmm_*_cmdstan.pkl"))
print(f"\nSaved HMM model files ({len(saved_models)}):")
for model_file in saved_models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name} ({size_mb:.2f} MB)")


TRAINING SUMMARY

Total models trained: 3
Models: ['hmm_2', 'hmm_3', 'hmm_4']

Saved in: ../models

Saved HMM model files (3):
  - hmm_2_cmdstan.pkl (46.32 MB)
  - hmm_3_cmdstan.pkl (54.37 MB)
  - hmm_4_cmdstan.pkl (62.48 MB)
